# Angular Center of Mass (ACoM) SIREN Training Pipeline
This notebook provides an interactive pipeline to generate a dataset of Centroidal Momentum Matrices (CMM), train a Sinusoidal Representation Network (SIREN) to approximate the Angular Center of Mass, and export the weights to C++ for Whole-Body MPC.

In [ ]:
import os
import jax
import jax.numpy as jnp
import numpy as np

# Ensure we are in the workspace root so paths resolve correctly
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

print(f"Working directory: {os.getcwd()}")
from humanoid_learning.acom.dataset_generator import AcomDatasetGenerator
from humanoid_learning.acom.train_acom import train_acom
from humanoid_learning.acom.export_acom import export_to_json, export_to_cpp_header
from humanoid_learning.acom.models import SirenAcom

## 1. Configuration & Data Collection
First, we specify the robot model and generate the training dataset.

In [ ]:
robot_name = "atlas"  # Options: "g1", "atlas"
xml_path = "robot_models/drc_atlas/drc_atlas_description/urdf/atlas.xml"

num_samples = 15000
print(f"Generating dataset for {robot_name.upper()}...")
generator = AcomDatasetGenerator(xml_path)
dataset = generator.generate_dataset(num_samples=num_samples)

print(f"Dataset generated! Shape of q_joints: {dataset['q_joints'].shape}")
print(f"Shape of Target A_bar_omega: {dataset['A_bar_omega'].shape}")

## 2. Training Monitoring
Launch TensorBoard to monitor the training loss and validation metrics.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /tmp/acom_export/tb_logs

## 3. Train the SIREN Network
Run the Jax optimization loop to train the network to approximate the connection 1-form.

In [ ]:
hidden_dim = 64
num_layers = 3
num_epochs = 50
log_dir = f"/tmp/acom_export/tb_logs/{robot_name}"

model, params, history = train_acom(
    dataset=dataset,
    in_dim=generator.num_joints,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    num_epochs=num_epochs,
    verbose=True,
    log_dir=log_dir
)

## 4. Evaluation & Sanity Checks
We can query the model to verify it outputs smooth orientation offsets and analytical Jacobians.

In [ ]:
# Pick a random configuration to evaluate
q_test = dataset['q_joints'][0:1]

# Forward pass to get orientation offset (Delta Theta)
theta_offset = model.apply({'params': params}, q_test)
print(f"Orientation offset for test config:\n {theta_offset}")

# Analytical Jacobian
jac_fn = jax.jacobian(lambda q: model.apply({'params': params}, q))
jacobian = jac_fn(q_test[0])
print(f"\nAnalytical Jacobian Shape (3 x n_j): {jacobian.shape}")

# Compare analytical Jacobian to Target A_bar_omega
target_jac = dataset['A_bar_omega'][0]
error = jnp.linalg.norm(jacobian - target_jac, ord='fro')
print(f"\nFrobenius Error vs Target: {error:.4f}")

## 5. Export to C++
Finally, we export the trained weights to JSON and directly to a C++ header file for use in the OCS2 MPC pipeline.

In [ ]:
output_dir = f"/home/nico-palomo/workspace/wb_humanoid_mpc/robot_models/drc_atlas/drc_atlas_centroidal_mpc/config/acom"
os.makedirs(output_dir, exist_ok=True)

json_path = os.path.join(output_dir, f"acom_{robot_name}.json")
cpp_path = os.path.join(output_dir, f"AcomSirenWeights.h")

export_to_json(params, json_path)
export_to_cpp_header(
    params, 
    cpp_path, 
    class_name=f"AcomSirenWeights"
)

print(f"✅ Export Complete!")
print(f"JSON: {json_path}")
print(f"C++ Header: {cpp_path}")

# Note: The C++ header should be moved to the humanoid_common_mpc/include/humanoid_common_mpc/acom/ directory
# and static compilation handles it automatically in AngularCenterOfMass.cpp